In [1]:
import boto3
import duckdb
import pandas as pd
import quilt3 as q3

# Step 1: Look up S3 path from Glue
glue = boto3.client('glue', region_name='us-east-1')
table = glue.get_table(DatabaseName='userathenadatabase-mbq1ihawbzb7', Name='titanic_merged_objects')
s3_path = table['Table']['StorageDescriptor']['Location']

# Step 2: Connect to DuckDB
con = duckdb.connect()

# Step 3: Load extensions
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL iceberg; LOAD iceberg;")
con.execute("CREATE SECRET (TYPE s3, PROVIDER credential_chain);")
con.execute("SET unsafe_enable_version_guessing = true;")

In [29]:
# Step 4: Fetch search results from Quilt
hits = [hit["_source"] for hit in q3.search("key_text:entry.json", limit=10000)]
search_df = pd.DataFrame(hits)
search_df

,delete_marker,key,last_modified,size,updated,version_id
0,False,benchling/test_entry/entry.json,2023-12-05T21:13:29.396Z,400,2023-12-05T21:13:37.187457,12BrEJFb6LSOZOpxt_iXvjClRubfpzQj
1,False,benchling/test_entry/entry.json,2023-12-05T19:44:55.056Z,400,2023-12-05T19:45:02.625462,Zb_SzBMpI74OPeIAKtTUJgoYyTcYX_Rx
2,False,glacier/quilt/catalog/node_modules/har-schema/...,2020-10-30T23:22:23+00:00,1031,2023-08-04T22:18:58.522325,D0fdT7axzbbSxafAUaU.FKKzfz_ycPVb
3,False,glacier/quilt/catalog/node_modules/file-entry-...,2020-10-30T23:22:09+00:00,3927,2023-08-04T22:18:55.400911,MOWVl_9upMXQwX7b7k8UlrqqxopTJvCQ
4,False,dima/node_modules4/file-entry-cache/package.json,2019-12-12T02:44:24+00:00,3921,2023-09-19T21:27:44.106905,sqPIkmd1DpfAFArUofUBe44.BRqAPe1Z
...,...,...,...,...,...,...
97,False,benchhook/etr_s9YzD3Lu/entry.json,2025-04-02T00:12:05.547Z,2841,2025-04-02T00:12:11.511121,ETpvhyCV5voHILilDJfGiYds_M2_8PeY
98,False,benchhook/etr_s9YzD3Lu/entry.json,2025-04-02T00:05:18.403Z,2841,2025-04-02T00:05:28.141944,pRn9kX8A.0OV6.BntN2txh4Uiuaga_W9
99,False,benchhook/etr_tY7Yuhzv/entry.json,2025-04-03T15:14:18.534Z,1127,2025-04-03T15:14:29.607320,pZrGL9nbtCPaGyQ8qJx7ECjFheHwt84U
100,True,"{typeFields=[""v2"",""entry"",""updated"",""reviewRec...",2025-03-01T00:32:28.375Z,0,2025-03-01T00:32:54.790102,sFc4eJLOoIgzs.9kUSbqiqQKftTfmJX7


In [25]:
pe_df = con.execute(f"""
    SELECT
        pkg_name,
        logical_key,
        regexp_extract(physical_key, '^s3://[^/]+/([^?]+)', 1) AS key_path,
        regexp_extract(physical_key, '[?&]versionId=([^&]+)', 1) AS version_id
    FROM iceberg_scan('{s3_path}', allow_moved_paths = true)
""").fetchdf()
pe_df

,pkg_name,logical_key,key_path,version_id
0,horizon/2024-04-12,README.md,horizon/2024-04-12/README.md,b5h97UnJKxDqWndGeDFFVbLOZlSGQ3_0
1,horizon/2024-04-12,stories/EP-2 Athena Works.md,horizon/2024-04-12/stories/EP-2%20Athena%20Wor...,bXoRnp.Ih92O5l4vkecnf9tOHEHLwMRk
2,horizon/2024-04-12,stories/EP-A-Email-Notifications.md,horizon/2024-04-12/stories/EP-A-Email-Notifica...,Rsk4pIgH1QmmgAw19i75Zk_vPsvX_hux
3,benchhook/etr_tY7Yuhzv,EP-BioIT2025-Social Banner.jpg,benchhook/etr_tY7Yuhzv/EP-BioIT2025-Social%20B...,Id.KQg15iyjQawToo833p0_9x06N00X6
4,benchhook/etr_tY7Yuhzv,demo3 2025-04-03 (etr_tY7Yuhzv).pdf,benchhook/etr_tY7Yuhzv/demo3%202025-04-03%20%2...,tIGb_vKoHwIQ4k2yjrLqNN7rT8SiK4ag
...,...,...,...,...
1778,benchling/etr_rv7MuD30,entry.json,benchling/etr_rv7MuD30/entry.json,WdNxPT0yh2q5F4xTIqvVwCvkWY6zqkEO
1779,benchling/etr_rv7MuD30,entry.json,benchling/etr_rv7MuD30/entry.json,WdNxPT0yh2q5F4xTIqvVwCvkWY6zqkEO
1780,benchhook/etr_0uKCQZ8f,entry.json,benchhook/etr_0uKCQZ8f/entry.json,hG9tmHFNPFUBGE20LTxKqMPhpRYRc5Mo
1781,benchhook/etr_0uKCQZ8f,quiltable 2025-03-24 (etr_0uKCQZ8f).pdf,benchhook/etr_0uKCQZ8f/quiltable%202025-03-24%...,R0F17hwS50JmqYZsO7xDxpfwzlXjEexq


In [33]:
con.register("es_results", search_df)

# Step 5: Query Iceberg table from S3 and join with search results
df = con.execute(f"""
    SELECT
        pe.*,
        es_results.key
    FROM iceberg_scan('{s3_path}', allow_moved_paths = true) AS pe
    JOIN es_results ON es_results.key = regexp_extract(pe.physical_key, '^s3://[^/]+/([^?]+)', 1)
""").fetchdf()


df

,pkg_name,top_hash,timestamp,logical_key,physical_key,size,hash,meta,source_bucket,key
0,benchhook/etr_s9YzD3Lu,25a4007fd64cbf9d104dbe06e745608b24cf8fd761e386...,1743547501,entry.json,s3://quilt-bake/benchhook/etr_s9YzD3Lu/entry.j...,2841,"{'type': 'sha2-256-chunked', 'value': 'vY1SC+r...",{},quilt-bake,benchhook/etr_s9YzD3Lu/entry.json
1,benchhook/etr_OtsAuzfT,76df621bb562794e8e1cd6c04bf170b7846b58429c167a...,1743536688,entry.json,s3://quilt-bake/benchhook/etr_OtsAuzfT/entry.j...,3581,"{'type': 'sha2-256-chunked', 'value': '8EPuLvg...",{},quilt-bake,benchhook/etr_OtsAuzfT/entry.json
2,benchhook/etr_GyOaiPoq,8d46f03eaaf7ecc62920891e2e8d4729ac7f497c1d66b4...,1743545041,entry.json,s3://quilt-bake/benchhook/etr_GyOaiPoq/entry.j...,1446,"{'type': 'sha2-256-chunked', 'value': '9egA9/5...",{},quilt-bake,benchhook/etr_GyOaiPoq/entry.json
3,benchhook/etr_pAsUM9zF,6fabac1bd763b78e50e35a4f887837884807a45200513f...,latest,entry.json,s3://quilt-bake/benchhook/etr_pAsUM9zF/entry.j...,3178,"{'type': 'sha2-256-chunked', 'value': '63mDJ79...",{},quilt-bake,benchhook/etr_pAsUM9zF/entry.json
4,benchhook/etr_0uKCQZ8f,3ca42279898b970c37fb93c66cbe92a814ceb6ca393de6...,1742998246,entry.json,s3://quilt-bake/benchhook/etr_0uKCQZ8f/entry.j...,3380,"{'type': 'sha2-256-chunked', 'value': 'JBPFiR5...",{},quilt-bake,benchhook/etr_0uKCQZ8f/entry.json
...,...,...,...,...,...,...,...,...,...,...
727,benchhook/etr_0uKCQZ8f,98d7183189a73363e13e76b1d47c6dbcd300beb9024189...,1742852102,entry.json,s3://quilt-bake/benchhook/etr_0uKCQZ8f/entry.j...,2518,"{'type': 'sha2-256-chunked', 'value': 'E46rQQA...",{},quilt-bake,benchhook/etr_0uKCQZ8f/entry.json
728,benchhook/etr_0uKCQZ8f,d0b54f7e2e213f9a5d59ba51cf2f4f552d7f85ff4f2607...,1743542244,entry.json,s3://quilt-bake/benchhook/etr_0uKCQZ8f/entry.j...,3624,"{'type': 'sha2-256-chunked', 'value': 'Kq0SDG3...",{},quilt-bake,benchhook/etr_0uKCQZ8f/entry.json
729,benchhook/etr_0uKCQZ8f,e337dea3db755c96eddee8af819634185d643c7904476d...,1742852100,entry.json,s3://quilt-bake/benchhook/etr_0uKCQZ8f/entry.j...,2518,"{'type': 'sha2-256-chunked', 'value': 'E46rQQA...",{},quilt-bake,benchhook/etr_0uKCQZ8f/entry.json
730,benchhook/etr_0uKCQZ8f,b4aa688ca772c1048045cc3887d81ff5af2fbe58cca351...,1743588676,entry.json,s3://quilt-bake/benchhook/etr_0uKCQZ8f/entry.j...,3490,"{'type': 'sha2-256-chunked', 'value': 'Vpu/rQS...",{},quilt-bake,benchhook/etr_0uKCQZ8f/entry.json


In [16]:
for pk in df['key_path']:
    print(pk)

benchling/etr_bl4xp2YJ/entry.json
benchling/etr_bl4xp2YJ/event_message.json
benchhook/etr_s9YzD3Lu/input.json
benchhook/etr_tY7Yuhzv/EP-BioIT2025-Social%20Banner.jpg
benchhook/etr_tY7Yuhzv/demo3%202025-04-03%20%28etr_tY7Yuhzv%29.pdf
benchhook/etr_GyOaiPoq/README.md
benchhook/etr_rv7MuD30/Nurix%20Demo%20Test%202024-09-26%20%28etr_rv7MuD30%29.pdf
benchhook/etr_rv7MuD30/entry.json
benchhook/etr_pAsUM9zF/README.md
test/2024-01-18/Template.md
